In [5]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import torch
from sklearn.metrics import accuracy_score

# 加载 IMDb 数据集
dataset = load_dataset("imdb")

# 初始化 BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# 数据预处理：将文本数据转换为 BERT 输入格式
def preprocess_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

# 对训练和测试集进行预处理
train_dataset = dataset['train'].map(preprocess_function, batched=True)
test_dataset = dataset['test'].map(preprocess_function, batched=True)

# BERT 模型
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# 定义训练参数
training_args = TrainingArguments(
    output_dir='./results',          # 输出目录
    learning_rate=2e-5,              # 学习率
    per_device_train_batch_size=16,  # 每个设备的训练批量大小
    per_device_eval_batch_size=64,   # 每个设备的评估批量大小
    num_train_epochs=3,              # 训练 3 个 epoch
    weight_decay=0.01,               # 权重衰减
    logging_dir='./logs',            # 日志目录
)
# 自定义评价指标
def compute_metrics(p):
    preds = torch.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds)}

# Trainer 初始化
trainer = Trainer(
    model=model,                         # 模型
    args=training_args,                  # 训练参数
    train_dataset=train_dataset,         # 训练数据集
    eval_dataset=test_dataset,           # 测试数据集
    compute_metrics=compute_metrics      # 评价指标
)

# 开始训练
trainer.train()

# 评估模型
eval_results = trainer.evaluate()

# 打印实验结果
print(f"Evaluation Results: {eval_results}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss


KeyboardInterrupt: 